*0.1 Python for GenAI*

# dotenv

**The situation.** A developer pastes the OpenAI key into `config.py` to get a demo working and pushes the branch to GitHub. GitHub's scanner flags it within minutes. Before anyone reads the alert, an automated script has found the key and spent $400. Deleting the file does not help — the key is in the commit history forever.

**The fix: secrets live in a file git never sees.** `.env` is a plain file of `NAME=value` lines in the project folder, listed in `.gitignore`. The program loads it into the *environment variables* — named values the operating system gives a program — and reads the key from there. In production there is no file: the platform (Kubernetes, AWS) sets the same names before the program starts. The code is identical in both places.

In [1]:
# Load OPENAI_API_KEY from the .env file. The OpenAI clients read it from the environment.
from dotenv import find_dotenv, load_dotenv

load_dotenv(find_dotenv())
MODEL = "gpt-4o-mini"

**Load it and read it.** `find_dotenv()` looks upward from the current folder for a `.env` file. `load_dotenv` reads it into the environment. This is the cell at the top of every notebook here.

In [2]:
import os
from pathlib import Path

from dotenv import find_dotenv, load_dotenv

env_path = find_dotenv()
print("found:", env_path.replace(str(Path.home()), "~"))
load_dotenv(env_path)
key = os.environ["OPENAI_API_KEY"]
print("OPENAI_API_KEY loaded:", key[:7] + "…" + key[-4:], "(never print the whole key)")
assert key.startswith("sk-")

found: ~/Documents/GenAI/GenAI_Learnings/GenAILayers/.env
OPENAI_API_KEY loaded: sk-proj…HH0A (never print the whole key)


**Reading the output.** The file was found in the project root and the key is in the environment, shown with the middle hidden.

**Per-environment files.** Staging and production have different settings. `dotenv_values` reads a file into a dictionary *without* touching the environment — useful for tooling and tests.

In [3]:
import tempfile

from dotenv import dotenv_values

with tempfile.TemporaryDirectory() as folder:
    staging = Path(folder) / ".env.staging"
    staging.write_text("APP_ENV=staging\nLOG_LEVEL=DEBUG\nOPENAI_MODEL=gpt-4o-mini\n")
    values = dotenv_values(staging)
print(".env.staging as a dictionary:", dict(values))
print("environment unchanged:", os.environ.get("APP_ENV"))
assert values["APP_ENV"] == "staging"

.env.staging as a dictionary: {'APP_ENV': 'staging', 'LOG_LEVEL': 'DEBUG', 'OPENAI_MODEL': 'gpt-4o-mini'}
environment unchanged: None


**The rule to remember.** A key in source code is a leaked key. Keys live in `.env` locally (git-ignored) and in the platform's secret store in production; the code reads the environment either way.

```
laptop        .env file (git-ignored) ──▶ environment variables ──▶ os.environ["OPENAI_API_KEY"]
production    Kubernetes secret / AWS Secrets Manager ──▶ environment variables ──▶ same line of code
```

| Use it when | Don't when | Instead use |
|---|---|---|
| local development and CI | production servers — the platform injects the values | a secret manager (Vault, AWS Secrets Manager) read at start-up |

**Watch out**
- Put `.env` in `.gitignore` *before* creating it. Commit a `.env.example` with fake values so teammates know the names.
- Never copy `.env` into a Docker image; the image goes to every registry and machine that pulls it.
- A key that was committed is already leaked. Revoke it and create a new one — rewriting history is not enough.